In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import os

In [ ]:
image_path_a = "samples/canvas.png"
image_a = cv2.imread(image_path_a)
image_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2RGB)
gray_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2GRAY)

image_path_b = "samples/b.png"
image_b = cv2.imread(image_path_b)
image_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2RGB)
gray_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_a, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
blurred_a = cv2.GaussianBlur(gray_a, (3, 3), 0)
plt.imshow(blurred_a, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
threshold_kernel_size = blurred_a.shape[0] // 32 * 2 + 1
thresh = cv2.adaptiveThreshold(
    blurred_a,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    threshold_kernel_size,
    3,
)
plt.imshow(thresh, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
edges = cv2.Canny(thresh, 50, 200, None, 3)

plt.imshow(edges, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Close gaps in the border
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

plt.imshow(edges, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

preview = image_a.copy()
cv2.drawContours(preview, contours, -1, (255, 0, 0), 2)
plt.imshow(preview)
plt.axis("off")
plt.show()

In [ ]:
# largest quadrilateral
board = None
for cnt in sorted(contours, key=cv2.contourArea, reverse=True):
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.04 * peri, True)

    if len(approx) == 4:
        board = approx.reshape(4, 2)

        tl = board[np.argmin(board.sum(axis=1))]
        br = board[np.argmax(board.sum(axis=1))]
        tr = board[np.argmin(np.diff(board, axis=1))]
        bl = board[np.argmax(np.diff(board, axis=1))]

        # Add margin to the bounding box
        margin = 10  # Adjust this value as needed
        tl = (tl[0] - margin, tl[1] - margin)
        br = (br[0] + margin, br[1] + margin)
        tr = (tr[0] + margin, tr[1] - margin)
        bl = (bl[0] - margin, bl[1] + margin)

        board = np.array([tl, tr, br, bl], dtype=np.int32)
        break

In [ ]:
if board is not None:
    # Order points in clockwise order
    rect = np.zeros((4, 2), dtype="float32")
    s = board.sum(axis=1)
    rect[0] = board[np.argmin(s)]
    rect[2] = board[np.argmax(s)]

    diff = np.diff(board, axis=1)
    rect[1] = board[np.argmin(diff)]
    rect[3] = board[np.argmax(diff)]

    (tl, tr, br, bl) = rect

    width_a = np.linalg.norm(br - bl)
    width_b = np.linalg.norm(tr - tl)
    max_width = max(int(width_a), int(width_b))

    height_a = np.linalg.norm(tr - br)
    height_b = np.linalg.norm(tl - bl)
    max_height = max(int(height_a), int(height_b))

    dst = np.array(
        [
            [0, 0],
            [max_width - 1, 0],
            [max_width - 1, max_height - 1],
            [0, max_height - 1],
        ],
        dtype="float32",
    )

    M = cv2.getPerspectiveTransform(rect, dst)
    warped_a = cv2.warpPerspective(image_a, M, (max_width, max_height))

    plt.imshow(warped_a)
    plt.axis("off")
    plt.show()

In [ ]:
blurred_warped_a = cv2.GaussianBlur(warped_a, (3, 3), 0)
plt.imshow(blurred_warped_a)
plt.axis("off")
plt.show()

In [ ]:
gray_warped_a = cv2.cvtColor(warped_a, cv2.COLOR_BGR2GRAY)

In [ ]:
threshold_kernel_size = blurred_warped_a.shape[0] // 8 * 2 + 1
thresh_warped = cv2.adaptiveThreshold(
    gray_warped_a,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    threshold_kernel_size,
    3,
)

# # Draw black border around the warped image to avoid false positives at the edges
# border_size = 1
# thresh_warped[:border_size, :] = 0
# thresh_warped[-border_size:, :] = 0
# thresh_warped[:, :border_size] = 0
# thresh_warped[:, -border_size:] = 0

plt.imshow(thresh_warped, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
edges = cv2.Canny(thresh_warped, 50, 150, apertureSize=3)

# Close gaps in the border
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)

plt.imshow(edges, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Detect hough lines in the warped image to draw grid
lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)

# Create black image to draw lines on grayscale 1 channel
line_image = np.ones_like(gray_warped_a) * 255

# Draw the detected lines on the warped image
if lines is not None:
    for rho, theta in lines[:, 0]:
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho
        x1 = int(x0 + 1000 * (-b))
        y1 = int(y0 + 1000 * (a))
        x2 = int(x0 - 1000 * (-b))
        y2 = int(y0 - 1000 * (a))
        cv2.line(line_image, (x1, y1), (x2, y2), (0, 0, 0), 2)


plt.imshow(line_image, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
# Line image is a binary image with grid lines
# Extract all the grid squares from the line image
contours, _ = cv2.findContours(line_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
rectangles = []
for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    aspect_ratio = w / h
    if (
        0.8 < aspect_ratio < 1.2 and w > 20 and h > 20
    ):  # Filter for square-like contours
        # Add small margin to the bounding box to avoid cutting off the edges of the cells
        margin = 5  # Adjust this value as needed
        x = max(x - margin, 0)
        y = max(y - margin, 0)
        w = min(w + 2 * margin, line_image.shape[1] - x)
        h = min(h + 2 * margin, line_image.shape[0] - y)
        rectangles.append((x, y, w, h))


# Sort the rectangles from left to right, top to bottom
rectangles.sort(key=lambda r: (r[1] // 50, r[0] // 50))

print(f"Found {len(rectangles)} rectangles")

# Extract all rectangles from the warped image to two dimensional list of cells
cells = [[] for _ in range(8)]
for x, y, w, h in rectangles:
    row = y // h
    col = x // w
    cell = warped_a[y : y + h, x : x + w]
    cells[row].append(cell)

In [ ]:
print(f"Extracted {sum(len(row) for row in cells)} cells")

In [ ]:
plt.imshow(cells[0][0])
plt.axis("off")
plt.show()

In [ ]:
# Remove the old images in the out directory
out_dir = "out"
if os.path.exists(out_dir):
    for file in os.listdir(out_dir):
        os.remove(os.path.join(out_dir, file))

# Write each cell to a separate image file in chess notation order (a1, a2, ..., h8)
for row in range(8):
    for col in range(8):
        cell = cells[row][col]
        # Chess notation: a1 is bottom-left, h8 is top-right
        chess_notation = f"{chr(ord('a') + col)}{8 - row}"
        out_path = os.path.join(out_dir, f"{chess_notation}.png")
        cv2.imwrite(out_path, cv2.cvtColor(cell, cv2.COLOR_RGB2BGR))